# Softmax Regression (Multi-class Logistic)
Train and tune a multinomial logistic regression model (Softmax Regression) for the 3-class diabetes target.

In [3]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
 )
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

USE_GPU = True
GPU_BACKEND = "cpu"
GPU_STATUS_MESSAGE = "CPU mode (cuML/CuPy not available)."

try:
    import cupy as cp
    from cuml.linear_model import LogisticRegression as cuLogisticRegression
    from cuml.preprocessing import StandardScaler as cuStandardScaler

    if cp.cuda.runtime.getDeviceCount() > 0:
        GPU_BACKEND = "gpu"
        dev_id = cp.cuda.runtime.getDevice()
        dev_name = cp.cuda.runtime.getDeviceProperties(dev_id)["name"].decode("utf-8")
        GPU_STATUS_MESSAGE = f"GPU mode enabled on CUDA device {dev_id}: {dev_name}"
except Exception as gpu_import_error:
    GPU_STATUS_MESSAGE = f"CPU mode (GPU libraries unavailable): {gpu_import_error}"

print(GPU_STATUS_MESSAGE)

CPU mode (GPU libraries unavailable): No type extension with name arrow.py_extension_type found


In [ ]:
dataset_dir = Path("datasets")
csv_paths = sorted(dataset_dir.rglob("*.csv"))

target_col = "Diabetes_012"
datasets = {}

for path in csv_paths:
    df_candidate = pd.read_csv(path)
    if target_col not in df_candidate.columns:
        print(f"Skipping {path}: missing target column '{target_col}'")
        continue
    datasets[path.as_posix()] = df_candidate

print(f"Loaded {len(datasets)} dataset(s):")
for name, data in datasets.items():
    print(f"- {name}: {data.shape}")

active_dataset_name = "datasets/BaseDataset.csv"
if active_dataset_name not in datasets:
    active_dataset_name = next(iter(datasets))

df = datasets[active_dataset_name].copy()
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

print(f"\nActive dataset: {active_dataset_name}")
print(f"Feature matrix shape: {X.shape}")
print("Class distribution:")
print(y.value_counts(normalize=True).sort_index().rename("proportion"))

In [ ]:
split_store = {}

for name, data in datasets.items():
    Xi = data.drop(columns=[target_col])
    yi = data[target_col].astype(int)

    # First hold out 20% test set, then carve 10% validation from the remaining 80%.
    Xi_train_val, Xi_test, yi_train_val, yi_test = train_test_split(
        Xi, yi, test_size=0.20, random_state=42, stratify=yi
    )
    Xi_train, Xi_val, yi_train, yi_val = train_test_split(
        Xi_train_val, yi_train_val, test_size=0.125, random_state=42, stratify=yi_train_val
    )

    split_store[name] = {
        "X_train": Xi_train,
        "X_val": Xi_val,
        "X_test": Xi_test,
        "y_train": yi_train,
        "y_val": yi_val,
        "y_test": yi_test,
    }

active_split = split_store[active_dataset_name]
X_train = active_split["X_train"]
X_val = active_split["X_val"]
X_test = active_split["X_test"]
y_train = active_split["y_train"]
y_val = active_split["y_val"]
y_test = active_split["y_test"]

def class_proportions(series):
    return series.value_counts(normalize=True).sort_index().rename("proportion")

print(
    f"Active split sizes ({active_dataset_name}): "
    f"Train={len(y_train)} ({len(y_train)/len(y):.2%}), "
    f"Validation={len(y_val)} ({len(y_val)/len(y):.2%}), "
    f"Test={len(y_test)} ({len(y_test)/len(y):.2%})"
)

print("\nClass distribution by split (active dataset):")
print("Original:")
print(class_proportions(y))
print("\nTrain:")
print(class_proportions(y_train))
print("\nValidation:")
print(class_proportions(y_val))
print("\nTest:")
print(class_proportions(y_test))

In [ ]:
# Train one Softmax model per dataset and select C by validation macro recall.
c_values = [0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
warnings.filterwarnings("ignore", category=ConvergenceWarning)

def _to_numpy(y_like):
    if GPU_BACKEND == "gpu":
        return cp.asnumpy(y_like)
    return np.asarray(y_like)

def fit_predict_for_c(X_train_df, y_train_sr, X_eval_df, c):
    if USE_GPU and GPU_BACKEND == "gpu":
        X_train_gpu = cp.asarray(X_train_df.to_numpy(dtype=np.float32))
        y_train_gpu = cp.asarray(y_train_sr.to_numpy(dtype=np.int32))
        X_eval_gpu = cp.asarray(X_eval_df.to_numpy(dtype=np.float32))

        scaler = cuStandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_gpu)
        X_eval_scaled = scaler.transform(X_eval_gpu)

        clf = cuLogisticRegression(
            penalty="l2",
            C=c,
            max_iter=2000,
            fit_intercept=True,
        )
        clf.fit(X_train_scaled, y_train_gpu)
        y_eval_pred = clf.predict(X_eval_scaled)

        model_bundle = {
            "backend": "gpu",
            "scaler": scaler,
            "model": clf,
        }
        return model_bundle, _to_numpy(y_eval_pred)

    cpu_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        (
            "softmax",
            LogisticRegression(
                multi_class="multinomial",
                solver="lbfgs",
                C=c,
                max_iter=2000,
                random_state=42,
            ),
        ),
    ])
    cpu_pipeline.fit(X_train_df, y_train_sr)
    y_eval_pred = cpu_pipeline.predict(X_eval_df)

    model_bundle = {
        "backend": "cpu",
        "model": cpu_pipeline,
    }
    return model_bundle, y_eval_pred

def predict_bundle(model_bundle, X_df):
    if model_bundle["backend"] == "gpu":
        X_gpu = cp.asarray(X_df.to_numpy(dtype=np.float32))
        X_scaled = model_bundle["scaler"].transform(X_gpu)
        y_pred = model_bundle["model"].predict(X_scaled)
        return _to_numpy(y_pred)
    return model_bundle["model"].predict(X_df)

trained_models = {}
validation_tables = {}

for name, split in split_store.items():
    Xi_train = split["X_train"]
    yi_train = split["y_train"]
    Xi_val = split["X_val"]
    yi_val = split["y_val"]

    val_scores = []
    for c in c_values:
        _, yi_val_pred = fit_predict_for_c(Xi_train, yi_train, Xi_val, c)
        val_recall_macro = recall_score(yi_val, yi_val_pred, average="macro")
        val_recall_weighted = recall_score(yi_val, yi_val_pred, average="weighted")
        val_acc = accuracy_score(yi_val, yi_val_pred)
        val_scores.append(
            {
                "C": c,
                "val_recall_macro": val_recall_macro,
                "val_recall_weighted": val_recall_weighted,
                "val_accuracy": val_acc,
            }
        )

    val_results = pd.DataFrame(val_scores).sort_values(
        ["val_recall_macro", "val_recall_weighted", "val_accuracy"],
        ascending=False,
    ).reset_index(drop=True)
    best_c = float(val_results.loc[0, "C"])

    X_train_final = pd.concat([Xi_train, Xi_val], axis=0)
    y_train_final = pd.concat([yi_train, yi_val], axis=0)

    # Refit best model on train+validation before final test evaluation.
    best_model_bundle, _ = fit_predict_for_c(X_train_final, y_train_final, Xi_val, best_c)

    trained_models[name] = {
        "bundle": best_model_bundle,
        "best_c": best_c,
        "backend": best_model_bundle["backend"],
    }
    validation_tables[name] = val_results

print(f"Execution backend: {GPU_BACKEND} (USE_GPU={USE_GPU})")
print(f"Trained {len(trained_models)} model(s).")
pd.DataFrame(
    [{"dataset": n, "best_C": v["best_c"], "backend": v["backend"]} for n, v in trained_models.items()]
).sort_values("dataset").reset_index(drop=True)

In [ ]:
# Validation leaderboard by dataset (top C per dataset by macro recall).
best_val_rows = []
for name, table in validation_tables.items():
    row0 = table.iloc[0].to_dict()
    best_val_rows.append(
        {
            "dataset": name,
            "best_C": row0["C"],
            "best_val_recall_macro": row0["val_recall_macro"],
            "best_val_recall_weighted": row0["val_recall_weighted"],
            "best_val_accuracy": row0["val_accuracy"],
        }
    )

val_leaderboard = pd.DataFrame(best_val_rows).sort_values("best_val_recall_macro", ascending=False).reset_index(drop=True)
val_leaderboard

In [ ]:
test_metrics_rows = []
reports_by_dataset = {}
conf_matrices = {}

for name, payload in trained_models.items():
    model_bundle = payload["bundle"]
    Xi_test = split_store[name]["X_test"]
    yi_test = split_store[name]["y_test"]

    yi_pred = predict_bundle(model_bundle, Xi_test)
    cm = confusion_matrix(yi_test, yi_pred, labels=[0, 1, 2])

    metrics_row = {
        "dataset": name,
        "backend": payload["backend"],
        "best_C": payload["best_c"],
        "test_accuracy": accuracy_score(yi_test, yi_pred),
        "test_recall_macro": recall_score(yi_test, yi_pred, average="macro"),
        "test_recall_weighted": recall_score(yi_test, yi_pred, average="weighted"),
        "test_f1_micro": f1_score(yi_test, yi_pred, average="micro"),
        "test_f1_macro": f1_score(yi_test, yi_pred, average="macro"),
        "test_f1_weighted": f1_score(yi_test, yi_pred, average="weighted"),
    }
    test_metrics_rows.append(metrics_row)
    reports_by_dataset[name] = classification_report(yi_test, yi_pred, digits=4)
    conf_matrices[name] = cm

test_metrics = pd.DataFrame(test_metrics_rows).sort_values("test_recall_macro", ascending=False).reset_index(drop=True)
test_metrics

In [ ]:
# Show classification reports.
for name in sorted(reports_by_dataset):
    print(f"\n{name}")
    print(reports_by_dataset[name])

# Plot a confusion matrix for each trained model.
for name in sorted(conf_matrices):
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrices[name], display_labels=[0, 1, 2])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.show()

## Metric Justification (Recall-Focused)
Because the diabetes classes are imbalanced and the goal is to reduce missed cases in underrepresented classes, `Macro-Recall` is the primary model-selection metric in this notebook.

- `Macro-Recall` gives each class equal weight, so minority-class sensitivity is not drowned out by majority classes.
- `Weighted-Recall` is still useful as a secondary check because it reflects support-weighted sensitivity.
- `Accuracy` and `Micro-F1` can look good even if minority classes have poor recall.
- `Macro-F1` remains important for balancing precision and recall, but when recall is the explicit priority, `Macro-Recall` should drive tuning decisions.